The following download is required for the code to work. The classic 'mediapipe.solutions' is not available anymore in the newer versions. So, along with 'uv add mediapipe', 'pose_landmarker.task' should be downloaded for the code to work.

In [1]:
import urllib.request
url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
urllib.request.urlretrieve(url, "models/pose_landmarker.task")

('models/pose_landmarker.task', <http.client.HTTPMessage at 0x114136ea550>)

In [2]:
import cv2
import mediapipe as mp
import pandas as pd
from pathlib import Path

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode
video_path = "../data/testvideo.mp4"

def extract_landmarks(video_path, output_csv, model_path="models/pose_landmarker.task"):
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.VIDEO
    )

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_idx = 0
    rows = []

    with PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int(frame_idx * (1000 / fps))

            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.pose_landmarks:
                for i, lm in enumerate(result.pose_landmarks[0]):
                    rows.append({
                        "frame": frame_idx, "landmark_id": i,
                        "x": lm.x, "y": lm.y, "z": lm.z,
                        "visibility": lm.visibility
                    })
            frame_idx += 1

    cap.release()
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    return df, fps

if __name__ == "__main__":
    extract_landmarks(video_path, "outputs/landmarks/testvideo.csv")

In [3]:
import cv2
import mediapipe as mp
from pathlib import Path

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Standard 33-point pose skeleton connections (same topology the old
# mp.solutions.pose.POSE_CONNECTIONS used) — defined manually since
# drawing_utils is no longer available in the Tasks API.
POSE_CONNECTIONS = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),  # arms + shoulders
    (11, 23), (12, 24), (23, 24),                       # torso
    (23, 25), (25, 27), (27, 29), (29, 31), (27, 31),   # left leg
    (24, 26), (26, 28), (28, 30), (30, 32), (28, 32),   # right leg
    (15, 17), (15, 19), (15, 21),                       # left hand
    (16, 18), (16, 20), (16, 22),                       # right hand
]

def preview_overlay(video_path, output_path, model_path="models/pose_landmarker.task"):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.VIDEO
    )

    cap = cv2.VideoCapture(video_path)
    w, h = int(cap.get(3)), int(cap.get(4))
    fps = cap.get(cv2.CAP_PROP_FPS)
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    print("Writer opened successfully:", out.isOpened())

    frame_idx = 0

    with PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int(frame_idx * (1000 / fps))

            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.pose_landmarks:
                landmarks = result.pose_landmarks[0]
                points = [(int(lm.x * w), int(lm.y * h)) for lm in landmarks]

                # draw connections
                for a, b in POSE_CONNECTIONS:
                    cv2.line(frame, points[a], points[b], (0, 255, 0), 2)

                # draw joints
                for x, y in points:
                    cv2.circle(frame, (x, y), 4, (0, 0, 255), -1)

            out.write(frame)
            frame_idx += 1

    cap.release()
    out.release()

preview_overlay(video_path, "outputs/videos/preview.mp4")

Writer opened successfully: True


In [4]:
import cv2
print(cv2.getBuildInformation())


General configuration for OpenCV 5.0.0 =====================================
  Version control:               5.0.0

  Platform:
    Timestamp:                   2026-07-01T09:07:32Z
    Host:                        Windows 10.0.20348 AMD64
    CMake:                       4.3.2
    CMake generator:             Visual Studio 17 2022
    CMake build tool:            C:/Program Files/Microsoft Visual Studio/2022/Enterprise/MSBuild/Current/Bin/amd64/MSBuild.exe
    MSVC:                        1944
    Configuration:               Debug Release
    Algorithm Hint:              ALGO_HINT_ACCURATE

  CPU/HW features:
    Baseline:                    SSE SSE2 SSE3
      requested:                 SSE3
    Dispatched code generation:  SSE4_1 SSE4_2 AVX FP16 AVX2 AVX512_SKX
      SSE4_1 (18 files):         + SSSE3 SSE4_1
      SSE4_2 (1 files):          + SSSE3 SSE4_1 POPCNT SSE4_2
      AVX (17 files):            + SSSE3 SSE4_1 POPCNT SSE4_2 AVX
      FP16 (0 files):            + SSSE3 SSE4_

In [5]:
video_path = "../data/testvideo.mp4"
cap = cv2.VideoCapture(video_path)
print("Input opened:", cap.isOpened())
w, h = int(cap.get(3)), int(cap.get(4))
fps = cap.get(cv2.CAP_PROP_FPS)
print("w, h, fps:", w, h, fps)

Input opened: True
w, h, fps: 1080 1920 29.97002997002997
